# 直接偏好优化（Direct Preference Optimization -- DPO）

RLHS有效，但是它要求训练三个模型（监督微调模型，奖励模型，策略模型），管理PPO的不稳定，以及调整KL惩罚。DPO问：是否可以把这些全部跳过？DPO直接在偏好对上对语言模型做优化，没有奖励模型，没有PPO。只有一个训练循环。

## 问题描述

RLHF的问题：
- 三个阶段各自需要一个模型，监督微调模型、奖励模型、策略模型
- 奖励模型需要成千上万的人类偏好数据对，以及一个单独的训练循环
- PPO 中的KL系数、学习率、裁剪率以及训练循环数都需要仔细调整

实践中，PPO的训练是出了名的不稳定。超参数的微小变化就可能导致训练法三。奖励模型也只是人类偏好的不完全代理，策略模型能够找到它对应的弱点。KL惩罚有用，但是用起来很困难。

DPO的关键洞察：你不需要一个单独的奖励模型。最优的奖励函数数学上由语言模型自身的词元概率所决定，你可以直接跳过奖励模型，然后让语言模型直接在偏好数据对上做优化。

DPO将RLHF压缩到只有一个监督学习步骤。一个模型，一个损失函数，一个训练循环，没有强化学习的过程。

## 基本概念

### 关键洞察

RLHF 的优化目标

```
maximize: E[R(x, y)] - beta * KL(pi || pi_ref)
```

DPO -- 对于任意的奖励函数R，最优的策略为：
```
pi*(y | x) = pi_ref(y | x) * exp(R(x, y) / beta) / Z(x)
```
拆分出R：
```
R(x, y) = beta * log(pi*(y | x) / pi_ref(y | x)) + beta * log Z(x)
```
`Z(x)`是一个归一化参数
将其带入到Bardley-Terry 偏好模型后约掉：
```
P(y_w > y_l | x) = sigmoid(R(x, y_w) - R(x, y_l))
 = sigmoid(beta * (log pi(y_w | x)/pi_ref(y_w | x) - log pi(y_l |x)/pi_ref(y_l | x)))
```
`_w` 是被偏好的回复win，`_l`是不被偏好的回复loss。
所以最后的概率就是：
模型相对于参考模型产生偏好答案的概率 - 模型相对于参考模型产生不被偏好答案的概率。优雅～

### 损失函数

```
L_DPO = -log(P(y_w > y_l | x))
```

### 对比RLHF 和 DPO

|视角|RLHF|DPO|
|---|---|---|
|需要训练的模型数|3（SFT+reward+policy）|1（policy only）|
|训练循环|3（SFT， RM， PPO）|2（SFT，DPO）|
|超参数|学习率，KL系数，裁剪率，奖励模型学习率...|学习率，beta|
|奖励模型|需要（单独训练）|隐式存在于概率|
|强化学习算法|PPO（复杂、不稳定）|监督学习（稳定）|
|GPU显存|显存中有3-4个模型|当前和推导两个模型|
|训练稳定性|超参数敏感|稳健，与SFT相似|

### 什么时候用DPO

1. 训练数据小；
2. 算力受限；
3. 快速迭代。

### 什么时候用RLHF

1. 训练数据大。主要原因是奖励模型可以捕捉更多的信息。
2. 当奖励信息很复杂的时候。比如更好由很多个维度决定。
3. 迭代式对齐。

### DPO的衍生

#### KTO

训练数据不需要好坏侧面同时出现，对回复标记“好”或“坏”即可

#### ORPO

将SFT和DPO两个训练步放在一起。通过修改损失函数实现

#### SimPO

彻底摆脱参考模型。直接使用模型产出的概率作为隐式奖励，不需要跟参考模型比较。

|方法|显存中的模型数|是否需要好坏成对数据|是否需要参考模型|几个训练循环|
|---|---|---|---|---|
|RLHF|3-4|Yes|Yes|3|
|DPO|2|Yes|Yes|2|
|KTO|2|No|Yes|2|
|ORPO|1|Yes|No|1|
|SimPO|1|Yes|No|1|